# FlyBot on AWS: YOLO26n fruit-fly detector on a Colab T4

Trains **YOLO26n** (Ultralytics) on the CamOS Roboflow dataset
[camos/fruit-fly-wmcqi](https://universe.roboflow.com/camos/fruit-fly-wmcqi) (**CC BY 4.0**) with the classes of
the version picked in the settings cell (v4: `fruit_fly` and `other_insect`, the bycatch the Lambda reports but never
counts), evaluates on the test split, exports **ONNX opset 17, 640x640 static, batch 1** with YOLO26's
**NMS-free end-to-end head** (output `[1, 300, 6]` = x1, y1, x2, y2, score, class), checks the graph on
**OpenCV 5 DNN (new engine) vs ONNX Runtime**, and downloads the files for `s3://sentinel-685527430355-models-public/yolo26/`.

1. Runtime > Change runtime type > **T4 GPU**
2. Fill the settings cell (a Roboflow API key downloads the dataset; without one the dataset cell asks for the zip)
3. Runtime > Run all. About 1 h on a T4 for the default epochs (not measured on YOLO26: the per-epoch time is
   printed by the first epochs; adjust `EPOCHS` so the run fits)
4. The last cell downloads: `fruitfly_yolo26n.onnx`, `fruitfly_yolo26n.onnx.json`, `fruitfly_yolo26n-test_metrics.json`,
   `fruitfly_yolo26n-results.csv`, `fruitfly_yolo26n-opencv_check.json`, `fruitfly_yolo26n-run.zip` and, when the
   end-to-end graph failed on OpenCV, `fruitfly_yolo26n-o2m.onnx` (one-to-many head, NMS in Python).

Then follow `models/COLAB_RUNBOOK.md` (upload to S3, `models/verify_yolo26.py` on the laptop, `DETECTOR=yolo26`).
Licence: Ultralytics YOLO26 is **AGPL-3.0**; the trained weights and this repository are AGPL-3.0 (non-commercial entry).

In [ ]:
# Settings
MODEL = 'yolo26n.pt'          # Ultralytics YOLO26 nano (NMS-free, end-to-end); fallback 'yolo11n.pt'
EPOCHS = 60                   # ~1 h on a T4 is the budget; the first epochs print the time per epoch: adjust
IMGSZ = 640
BATCH = 32
NAME = 'fruitfly_yolo26n'
VERSION = '2026-10-29'        # goes into the .onnx.json sidecar and the contract's model.version
DATASET_ZIP = '/content/drive/MyDrive/camos-fruitfly/fruit-fly.v4i.yolov8.zip'   # used when ROBOFLOW_API_KEY is empty; a /content/drive path mounts Google Drive, anything else prompts for an upload
OUT_DIR = '/content/drive/MyDrive/camos-fruitfly/yolo26-out'   # '' downloads the files through the browser instead of copying them to Drive
ROBOFLOW_API_KEY = ''         # universe.roboflow.com > settings > API key; downloads the version below (yolov8 export)
ROBOFLOW = ('camos', 'fruit-fly-wmcqi', 4)   # v3: fruit_fly only; v4: + other_insect (ants, Drosophila)
DATASET = f'{ROBOFLOW[0]}/{ROBOFLOW[1]} v{ROBOFLOW[2]} (CC BY 4.0)'
SYNTH_ZIP = ''                # optional: data/synth/trainset-yolo.zip (eval.synth.trainset, SIMULATED flies on empty trap photos), added to train only; a /content/drive path mounts Drive, 'upload' prompts
SMOKE_TEST = False            # True: 1 epoch on 5% of train, to check the notebook end to end in minutes
ULTRALYTICS = '8.4.177'       # same version as models/uv.lock; YOLO26 needs >= 8.3.200 (adjust if pip refuses)
OPENCV = '5.0.0.93'           # the wheel the Lambda image uses (opencv-python-headless)

In [ ]:
!nvidia-smi -L
!pip -q install ultralytics=={ULTRALYTICS} onnx onnxslim onnxruntime roboflow
# Colab preinstalls OpenCV 4.x and ultralytics pulls opencv-python 4.x: remove every cv2 package, then install the 5.x headless wheel
!pip -q uninstall -y opencv-python opencv-python-headless opencv-contrib-python opencv-contrib-python-headless
!pip -q install opencv-python-headless=={OPENCV} || pip -q install 'opencv-python-headless>=5,<6'
import cv2, ultralytics
print('ultralytics', ultralytics.__version__, 'opencv', cv2.__version__)
assert cv2.__version__.startswith('5.'), 'the OpenCV check below needs OpenCV 5: run "pip install opencv-python-headless>=5,<6" and restart the runtime'

In [ ]:
# Dataset: Roboflow download (API key) or the uploaded zip; polygon rows -> boxes; absolute data.yaml
import os, shutil
from pathlib import Path
import yaml

DATA = Path('/content/data')
if not (DATA / 'train').exists():
    if ROBOFLOW_API_KEY:
        from roboflow import Roboflow
        ws, proj, ver = ROBOFLOW
        ds = Roboflow(api_key=ROBOFLOW_API_KEY).workspace(ws).project(proj).version(ver).download('yolov8', location=str(DATA))
        print('downloaded', ds.location)
    else:
        if DATASET_ZIP.startswith('/content/drive/') and not os.path.ismount('/content/drive'):
            from google.colab import drive
            drive.mount('/content/drive')
        if not os.path.exists(DATASET_ZIP):
            from google.colab import files
            uploaded = files.upload()  # pick the version's yolov8 zip
            DATASET_ZIP = '/content/' + next(iter(uploaded))
        shutil.unpack_archive(DATASET_ZIP, DATA)


def polygon_rows_to_boxes(label_dir):
    # 31 train files hold polygon rows, 3 of them mixed with box rows (ultralytics drops those as corrupt)
    rows = 0
    for txt in Path(label_dir).glob('*.txt'):
        out, changed = [], False
        for line in txt.read_text().splitlines():
            p = line.split()
            if len(p) >= 7 and len(p) % 2 == 1:
                xs, ys = [float(v) for v in p[1::2]], [float(v) for v in p[2::2]]
                x0, x1, y0, y1 = max(0, min(xs)), min(1, max(xs)), max(0, min(ys)), min(1, max(ys))
                line = f'{p[0]} {(x0 + x1) / 2:.6f} {(y0 + y1) / 2:.6f} {x1 - x0:.6f} {y1 - y0:.6f}'
                rows, changed = rows + 1, True
            out.append(line)
        if changed:
            txt.write_text('\n'.join(out) + '\n')
    return rows


for split in ['train', 'valid', 'test']:
    n = len(list((DATA / split / 'images').glob('*')))
    print(split, n, 'images,', polygon_rows_to_boxes(DATA / split / 'labels'), 'polygon rows -> boxes')
# Class names in the export's order (class id = index); the detector counts the fruit_fly class only
NAMES = list(yaml.safe_load((DATA / 'data.yaml').read_text())['names'])
assert 'fruit_fly' in NAMES, NAMES
print('classes', NAMES)
train = ['train/images']
if SYNTH_ZIP:  # synthetic photos join the training split only; valid and test stay the dataset's own
    if SYNTH_ZIP.startswith('/content/drive/') and not os.path.ismount('/content/drive'):
        from google.colab import drive
        drive.mount('/content/drive')
    if not os.path.exists(SYNTH_ZIP):
        from google.colab import files
        SYNTH_ZIP = '/content/' + next(iter(files.upload()))
    shutil.unpack_archive(SYNTH_ZIP, '/content/synth')
    train.append('/content/synth/images')
    print('synthetic', len(list(Path('/content/synth/images').glob('*.jpg'))), 'images (class 0 =', NAMES[0], ')')
    assert NAMES[0] == 'fruit_fly', 'the synthetic labels use class 0 for fruit_fly'
data_yaml = {'path': str(DATA), 'train': train if len(train) > 1 else train[0], 'val': 'valid/images',
             'test': 'test/images', 'nc': len(NAMES), 'names': NAMES}
Path('/content/data.yaml').write_text(yaml.safe_dump(data_yaml, sort_keys=False))

In [ ]:
# Train (the per-epoch time printed by the first epochs x EPOCHS = ETA; keep it near 1 h on a T4)
from ultralytics import YOLO, settings

settings.update({'sync': False})
model = YOLO(MODEL)
model.train(data='/content/data.yaml', epochs=1 if SMOKE_TEST else EPOCHS, imgsz=IMGSZ, batch=BATCH,
            fraction=0.05 if SMOKE_TEST else 1.0, device=0, workers=4, patience=20, seed=0,
            deterministic=True, cache='ram', project='/content/runs', name=NAME, exist_ok=True)
RUN = Path(model.trainer.save_dir)
BEST = RUN / 'weights' / 'best.pt'
print('best weights', BEST, BEST.stat().st_size, 'bytes')

In [ ]:
# Test split: fruit_fly mAP50 is the acceptance number (YOLOv7-tiny baseline: 0.958, models/MODEL_CARD.md)
import json

r = YOLO(str(BEST)).val(data='/content/data.yaml', split='test', imgsz=IMGSZ, batch=16, device=0,
                        project=str(RUN), name='test', exist_ok=True)
metrics = {'model': MODEL, 'split': 'test', 'imgsz': IMGSZ, 'mAP50': round(float(r.box.map50), 4),
           'mAP50_95': round(float(r.box.map), 4), 'precision': round(float(r.box.mp), 4),
           'recall': round(float(r.box.mr), 4), 'smoke_run': SMOKE_TEST, 'epochs': EPOCHS,
           'ultralytics': ultralytics.__version__, 'dataset': DATASET,
           'per_class': {NAMES[int(c)]: {'mAP50': round(float(r.box.ap50[i]), 4), 'mAP50_95': round(float(r.box.ap[i]), 4)}
                         for i, c in enumerate(r.box.ap_class_index)}}
(RUN / 'test_metrics.json').write_text(json.dumps(metrics, indent=2))
print(metrics)

In [ ]:
# ONNX for OpenCV 5 DNN: opset 17, 640 static, batch 1, simplified. YOLO26 exports its NMS-free end-to-end head
# by default (output [1, 300, 6]); a one-to-many copy ([1, 4+nc, 8400], NMS in Python) is exported too as the fallback.
import onnx


def export(end2end: bool, suffix: str):
    m = YOLO(str(BEST))
    # the export flag picks the head (toggling head.end2end on the model is ignored by the exporter)
    path = m.export(format='onnx', opset=17, imgsz=IMGSZ, dynamic=False, simplify=True, batch=1, device='cpu', end2end=end2end)
    dst = f'/content/{NAME}{suffix}.onnx'
    shutil.copy(path, dst)
    g = onnx.load(dst)
    shapes = [[d.dim_value for d in o.type.tensor_type.shape.dim] for o in g.graph.output]
    ops = sorted({n.op_type for n in g.graph.node})
    print(dst, 'outputs', shapes, 'opset', [i.version for i in g.opset_import], 'ops', ops)
    return dst, shapes, ops


ONNX_E2E, SHAPES_E2E, OPS_E2E = export(True, '')
ONNX_O2M, SHAPES_O2M, OPS_O2M = export(False, '-o2m')
assert SHAPES_E2E[0][-1] == 6, f'end-to-end export should be [1, N, 6], got {SHAPES_E2E}'
assert SHAPES_O2M[0][-1] != 6, f'one-to-many export should be [1, 4+nc, A], got {SHAPES_O2M}'

In [ ]:
# Day-1 check: cv2.dnn (OpenCV 5, new engine) must load the graph and match ONNX Runtime on the test images.
# Same preprocessing as lambda/vision/sentinel/vision/detect_yolov7.preprocess (letterbox 640, pad 114, RGB, /255).
import numpy as np, onnxruntime as ort, time, glob


def letterbox(img, size=640, pad=114):
    h, w = img.shape[:2]
    s = min(size / h, size / w)
    nh, nw = round(h * s), round(w * s)
    resized = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)
    out = np.full((size, size, 3), pad, np.uint8)
    top, left = (size - nh) // 2, (size - nw) // 2
    out[top:top + nh, left:left + nw] = resized
    return out


def check(onnx_path, n=20):
    rep = {'onnx': onnx_path, 'opencv': cv2.__version__, 'onnxruntime': ort.__version__, 'photos': [], 'status': None}
    try:
        net = cv2.dnn.readNet(onnx_path, '', '', cv2.dnn.ENGINE_NEW)
    except cv2.error as e:
        rep.update(status='opencv_cannot_load', error=str(e)[:800]); return rep
    sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
    name = sess.get_inputs()[0].name
    worst = 0.0
    for p in sorted(glob.glob(str(DATA / 'test' / 'images' / '*')))[:n]:
        img = cv2.imread(p)
        blob = cv2.dnn.blobFromImage(letterbox(img, IMGSZ), 1 / 255.0, (IMGSZ, IMGSZ), swapRB=True, crop=False)
        try:
            net.setInput(blob); t0 = time.perf_counter(); out_cv = net.forward(); ms = (time.perf_counter() - t0) * 1000
        except cv2.error as e:
            rep.update(status='opencv_cannot_run', error=str(e)[:800]); return rep
        out_ort = sess.run(None, {name: blob})[0]
        same = out_cv.shape == out_ort.shape
        if same and out_cv.shape[-1] == 6:
            # end-to-end head: compare detections above 0.25, not raw rows (TopK orders tied low-score rows per engine)
            a_, b_ = out_cv.reshape(-1, 6), out_ort.reshape(-1, 6)
            a_, b_ = a_[a_[:, 4] > 0.25], b_[b_[:, 4] > 0.25]
            a_, b_ = a_[np.argsort(-a_[:, 4])], b_[np.argsort(-b_[:, 4])]
            diff = float(np.abs(a_ - b_).max()) if a_.shape == b_.shape and len(a_) else (0.0 if a_.shape == b_.shape else None)
        else:
            diff = float(np.abs(out_cv - out_ort).max()) if same else None
        worst = max(worst, diff or 0.0)
        rep['photos'].append({'image': Path(p).name, 'shape_cv': list(out_cv.shape), 'shape_ort': list(out_ort.shape),
                              'max_abs_diff': diff, 'finite': bool(np.isfinite(out_cv).all()), 'opencv_forward_ms': round(ms, 1)})
    rep.update(status='ok' if all(r['max_abs_diff'] is not None and r['max_abs_diff'] < 0.5 and r['finite'] for r in rep['photos']) else 'mismatch',
               worst_max_abs_diff=worst, photos_checked=len(rep['photos']))
    return rep


CHECK = {'end2end': check(ONNX_E2E), 'one_to_many': check(ONNX_O2M)}
for k, v in CHECK.items():
    print(k, v['status'], v.get('error', ''), 'worst diff', v.get('worst_max_abs_diff'))
Path(f'/content/{NAME}-opencv_check.json').write_text(json.dumps(CHECK, indent=2))
DEPLOY = ONNX_E2E if CHECK['end2end']['status'] == 'ok' else ONNX_O2M
HEAD = 'end2end' if DEPLOY == ONNX_E2E else 'one_to_many'
print('deploy:', DEPLOY, HEAD)
if HEAD == 'one_to_many':
    print('FALLBACK: the end-to-end head did not run on OpenCV 5 DNN; deploying the one-to-many graph (NMS in Python, '
          'lambda/vision/sentinel/vision/detect_yolo26.decode_one_to_many). Record this in docs/DECISIONS.md.')

In [ ]:
# Sidecar JSON (read by the detector: imgsz, head, version, provenance) + checksum, then download everything
import hashlib

final = f'/content/{NAME}.onnx'
if DEPLOY != final:
    shutil.copy(DEPLOY, final)
sha = hashlib.sha256(Path(final).read_bytes()).hexdigest()
side = {'name': 'yolo26n', 'version': VERSION, 'imgsz': IMGSZ, 'head': HEAD, 'classes': NAMES, 'opset': 17,
        'sha256': sha, 'bytes': Path(final).stat().st_size, 'trained': metrics, 'ultralytics': ultralytics.__version__,
        'opencv_check': {k: v['status'] for k, v in CHECK.items()}, 'license': 'AGPL-3.0 (Ultralytics YOLO26)',
        'dataset': DATASET, 'preprocess': 'letterbox 640 pad 114, RGB, /255'}
Path(f'/content/{NAME}.onnx.json').write_text(json.dumps(side, indent=2))
print(json.dumps(side, indent=2))

shutil.copy(RUN / 'results.csv', f'/content/{NAME}-results.csv')
shutil.copy(RUN / 'test_metrics.json', f'/content/{NAME}-test_metrics.json')
shutil.make_archive(f'/content/{NAME}-run', 'zip', RUN)
OUTPUTS = [f'{NAME}.onnx', f'{NAME}.onnx.json', f'{NAME}-test_metrics.json', f'{NAME}-results.csv',
           f'{NAME}-opencv_check.json', f'{NAME}-run.zip'] + ([f'{NAME}-o2m.onnx'] if HEAD == 'end2end' else [])
if OUT_DIR:
    if OUT_DIR.startswith('/content/drive/') and not os.path.ismount('/content/drive'):
        from google.colab import drive
        drive.mount('/content/drive')
    os.makedirs(OUT_DIR, exist_ok=True)
    for f in OUTPUTS:
        shutil.copy('/content/' + f, OUT_DIR)
    print('copied to', OUT_DIR, sorted(os.listdir(OUT_DIR)))
else:
    from google.colab import files
    for f in OUTPUTS:
        files.download('/content/' + f)
print('SHA-256', sha)

## After the run (on the laptop)

```
# put the downloaded files in models/yolo26/ (git-ignored), then:
bash models/upload_yolo26.sh                     # -> s3://sentinel-685527430355-models-public/yolo26/ (profile sentinel)
cd lambda/vision && uv run --with onnxruntime python ../../models/verify_yolo26.py ../../models/yolo26/fruitfly_yolo26n.onnx
DETECTOR=yolo26 make replay-sample               # 24 sample photos through the handler with YOLO26
```
Copy the test-split numbers into `models/MODEL_CARD.md` from `fruitfly_yolo26n-test_metrics.json` (cite the file);
set `Detector=yolo26` + the SHA-256 in `infra/deploy.sh` (or `DETECTOR=yolo26 make deploy`).